In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import lamindb as ln
from scipy.stats import binom, poisson
from sklearn.neighbors import NearestNeighbors

ln.track("uKmRjpiq5gvz")

In [ ]:
analysis_label  = ln.ULabel(name='fh1_pretreatment').save()
timepoint_label = ln.ULabel(name='MM Pre-Treatment').save()
qc_label        = ln.ULabel(name='qc').save()
cutoff_label    = ln.ULabel(name='cutoff').save()
sweep_label     = ln.ULabel(name='sweep').save()
filtered_label  = ln.ULabel(name='filtered').save()

In [ ]:
FIGURES_DIR = '/home/workspace/PCD_data/xenium/fh1_pretreatment/figures/qc_filtering_enrichment'

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
def save_fig(fig, stem, description, ulabels=(), features=None):
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    for ext in ['png', 'pdf']:
        path = os.path.join(FIGURES_DIR, f'{stem}.{ext}')
        fig.savefig(path, dpi=150, bbox_inches='tight')
        artifact = ln.Artifact(path, key=f'{stem}.{ext}', description=description).save()
        if ulabels:
            artifact.ulabels.add(*ulabels)
        if features:
            artifact.features.add_values(features)
    plt.close(fig)

In [ ]:
SAMPLE_NAMES = [
    'TIS04399-001',
    'TIS02434-001',
    'TIS02437-001',
    'TIS04398-001',
    'TIS02611-001',
    'TIS02603-001',
    'TIS02604-001',
    'TIS02613-001',
    'TIS02421-001',
    'TIS02430-001',
    'TIS02424-001',
    'TIS02439-001',
    'TIS02594-001',
]

QC_COLUMNS = ['total_counts', 'n_genes_by_counts']

OUTS_DIR = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data/outs'

NOISE_QUANTILE = 0.999

In [ ]:
ln.Param(name='cutoff_method', dtype=str).save()
ln.Feature(name='n_samples', dtype=int).save()
ln.Feature(name='noise_quantile', dtype=float).save()
ln.Feature(name='previous_min_counts', dtype=int).save()
ln.Feature(name='previous_min_genes', dtype=int).save()
ln.Feature(name='neighbors', dtype=int).save()
ln.Feature(name='sparse_fraction', dtype=float).save()
ln.Feature(name='candidate_min_counts_max', dtype=int).save()
ln.Feature(name='n_cells_before', dtype=int).save()
ln.Feature(name='n_cells_after', dtype=int).save()
ln.Feature(name='panel_genes', dtype=int).save()

shared_features = {
    'cutoff_method': 'sparse_tissue_enrichment',
    'n_samples': len(SAMPLE_NAMES),
    'noise_quantile': NOISE_QUANTILE,
}

In [ ]:
samples = {}
qc      = {}
spatial = {}
for name in SAMPLE_NAMES:
    adata = ln.Artifact.get(key=f'{name}.h5ad').load()
    # adds obs['n_genes_by_counts'] and the log1p columns, overwrites obs['total_counts']; .X unchanged, in memory only
    sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
    samples[name] = adata
    qc[name] = adata.obs[['total_counts', 'n_genes_by_counts', 'cell_area']].rename(columns={'n_genes_by_counts': 'n_genes'})
    spatial[name] = adata.obsm['spatial']

In [ ]:
fig, axes = plt.subplots(
    len(samples), len(QC_COLUMNS),
    figsize=(5.5 * len(QC_COLUMNS), 5 * len(samples)),
    squeeze=False,
)

for i, (name, adata) in enumerate(samples.items()):
    for j, column in enumerate(QC_COLUMNS):
        ax = axes[i, j]
        sc.pl.spatial(
            adata, color=column, vmax='p99', spot_size=50,
            ax=ax, show=False, colorbar_loc=None,
            title=f'{name} — {column}\n(median {adata.obs[column].median():.0f})',
        )
        fig.colorbar(
            ax.collections[0],
            cax=ax.inset_axes([1.03, 0, 0.03, 1]),
        )

fig.tight_layout()
plt.show()

In [ ]:
ncols = 3
nrows = -(-len(samples) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = axes.flatten()

for ax, (name, adata) in zip(axes, samples.items()):
    sc.pl.highest_expr_genes(adata, n_top=10, ax=ax, show=False)
    ax.set_title(name)

for ax in axes[len(samples):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
PREVIOUS_MIN_COUNTS = 15
PREVIOUS_MIN_GENES  = 10

In [ ]:
metrics = pd.concat(
    {name: pd.read_csv(f'{OUTS_DIR}/{name}/metrics_summary.csv').iloc[0] for name in SAMPLE_NAMES},
    axis=1,
).T
noise_rate = metrics.loc[SAMPLE_NAMES, 'estimated_number_of_false_positive_transcripts_per_cell'].astype(float)
noise_min_counts = pd.Series(poisson.ppf(NOISE_QUANTILE, noise_rate) + 1, index=SAMPLE_NAMES)

pd.DataFrame({'false_positives_per_cell': noise_rate, 'noise_min_counts': noise_min_counts})

In [ ]:
CANDIDATE_MIN_COUNTS = np.arange(1, 61)
NEIGHBORS            = 10
SPARSE_FRACTION      = 0.10

STEP_WIDTH         = 5
MIN_CELLS_PER_STEP = 30
ALPHA              = 0.05

first_quiet_step_start = {}
for name in SAMPLE_NAMES:
    coordinates = spatial[name]
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates).kneighbors(coordinates)[0][:, -1]
    in_sparse_tissue  = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)
    counts = qc[name]['total_counts'].to_numpy()

    first_quiet_step_start[name] = np.nan
    for step_start in range(0, int(CANDIDATE_MIN_COUNTS.max()), STEP_WIDTH):
        in_step        = (counts >= step_start) & (counts < step_start + STEP_WIDTH)
        cells_in_step  = in_step.sum()
        lonely_in_step = (in_step & in_sparse_tissue).sum()
        p_value = binom.sf(lonely_in_step - 1, cells_in_step, SPARSE_FRACTION)
        if cells_in_step >= MIN_CELLS_PER_STEP and p_value >= ALPHA:
            first_quiet_step_start[name] = step_start
            break

cutoffs = pd.DataFrame({'noise_min_counts': noise_min_counts})
# adds cutoffs['enrichment_min_counts'], floored at the noise floor; in memory only
cutoffs['enrichment_min_counts'] = np.maximum(pd.Series(first_quiet_step_start), noise_min_counts)

cutoffs

In [ ]:
ncols = 3
nrows = -(-len(SAMPLE_NAMES) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
for ax, name in zip(axes.ravel(), SAMPLE_NAMES):
    coordinates = spatial[name]
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates).kneighbors(coordinates)[0][:, -1]
    in_sparse_tissue  = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)

    counts  = qc[name]['total_counts'].to_numpy()
    removed = counts[:, None] < CANDIDATE_MIN_COUNTS[None, :]
    removed_per_candidate = np.maximum(removed.sum(axis=0), 1)
    enrichment = (removed & in_sparse_tissue[:, None]).sum(axis=0) / removed_per_candidate / SPARSE_FRACTION

    ax.plot(CANDIDATE_MIN_COUNTS, enrichment, color='tab:blue')
    ax.axhline(1, color='gray', ls=':', label='no enrichment (1)')
    ax.set_xlabel('minimum counts')
    ax.set_ylabel('enrichment in sparse tissue (fold)', color='tab:blue')
    ax.set_title(name)

    fraction_axis = ax.twinx()
    fraction_axis.plot(CANDIDATE_MIN_COUNTS, removed.mean(axis=0), color='tab:orange')
    fraction_axis.set_ylabel('fraction of cells removed', color='tab:orange')

    ax.axvline(PREVIOUS_MIN_COUNTS, color='tab:red', ls='--', label=f'previous ({PREVIOUS_MIN_COUNTS})')
    if not np.isnan(cutoffs.loc[name, 'enrichment_min_counts']):
        ax.axvline(cutoffs.loc[name, 'enrichment_min_counts'], color='tab:purple', ls='-.', label=f"enrichment ({cutoffs.loc[name, 'enrichment_min_counts']:.0f})")
    ax.legend(fontsize=8, loc='center right')

for ax in axes.ravel()[len(SAMPLE_NAMES):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

# save_fig(fig, 'fh1_pretreatment_qc_enrichment_cutoff_sweep', 'enrichment of removed cells in sparse tissue across candidate count cutoffs, with previous and enrichment cutoffs',
#          ulabels=(analysis_label, timepoint_label, qc_label, cutoff_label, sweep_label),
#          features={**shared_features, 'previous_min_counts': PREVIOUS_MIN_COUNTS, 'neighbors': NEIGHBORS,
#                    'sparse_fraction': SPARSE_FRACTION, 'candidate_min_counts_max': int(CANDIDATE_MIN_COUNTS.max())})

In [ ]:
filtered = {}
for name in SAMPLE_NAMES:
    adata = samples[name]
    keep = (
        (adata.obs['total_counts'] >= PREVIOUS_MIN_COUNTS)
        & (adata.obs['n_genes_by_counts'] >= PREVIOUS_MIN_GENES)
    )
    # drops cells below the default cutoffs; in memory only
    filtered[name] = adata[keep.to_numpy()].copy()

# adds obs['sample']; in memory only
combined = sc.concat(filtered, label='sample')

# drops AZU1 from .X and .var; in memory only
combined = combined[:, combined.var_names != 'AZU1'].copy()

# adds layers['counts'] (raw counts, since normalize_total overwrites .X)
combined.layers['counts'] = combined.X.copy()

combined

In [ ]:
# writes the combined object as a lamindb artifact
artifact = ln.Artifact.from_anndata(combined, key='fh1_pretreatment_combined_filtered_default_cutoffs_noAZU1.h5ad',
                                    description='Combined FH1 pretreatment Xenium object after per-sample QC filtering at the default cutoffs (15 counts, 10 genes), AZU1 removed').save()
artifact.features.add_values({
    'cutoff_method': 'fixed_default',
    'n_samples': len(SAMPLE_NAMES),
    'previous_min_counts': PREVIOUS_MIN_COUNTS,
    'previous_min_genes': PREVIOUS_MIN_GENES,
    'n_cells_before': sum(adata.n_obs for adata in samples.values()),
    'n_cells_after': combined.n_obs,
    'panel_genes': combined.n_vars,
})
artifact.ulabels.add(analysis_label, timepoint_label, qc_label, cutoff_label, filtered_label)

ln.finish()